# Compare the lists of available objects from different Oracle instances
* DEV1 / End-to-End
* STG1 / Development
* DEV3 / Dress Rehearsal
* PRD1 / Production

In [0]:
debug = False

In [0]:
import os, json, sys
from pyspark.sql import functions as fn

# Add the workspace path to sys.path
sys.path.append('/Workspace/Users/ab092898@flooranddecor.com/databricks-entdatalakehouse')
from UTILS import environment

In [0]:
env = environment.currentEnv(spark)

# Scan for files
files = os.scandir('/Volumes/bronze_dev/oracle-test/oracle-test/')

jsons = []

# Get list of those ending in .json; for name, cut off the mfcs_ start and .json end
for f in [f for f in files if f.name.endswith('.json')]:
    jsons.append({'path': f.path, 'name': f.name[5:-5]})

if debug:
    display(jsons)

## Read the contents of the files into list of data frames

In [0]:
# Loop over those files and read them in
for j in jsons:
    with open(j['path'], 'r') as file:
        objs = json.load(file)

    # Each file is rows of results - extract the items list from these
    for i in objs['results']:
        # Shred, and add the source moniker (cut off mfcs_ start and .json end)
        j['data'] = spark.createDataFrame(i['items'])
        
        if debug:
            # Sample some rows if desired
            display(so.take(7))

## Get a superset list of every known object...

In [0]:
selcols = ['owner', 'obj', 'object_type', 'num_rows']
joincols = ['owner', 'obj', 'object_type']

# Get desired columns from first data frame
superset = jsons[0]['data'].select(joincols)

# Union the same from the remaining data frames
for so in jsons[1:]:
    superset = superset.union(so['data'].select(joincols))

# Distinctify the values
superset = superset.distinct()

if debug:
    display(superset)

## ... and a common set of tables, all of which exist in each environment

In [0]:
# Get desired columns from first data frame
common = jsons[0]['data'].select(joincols)

# Join the remaining data frames
for so in jsons[1:]:
    common = common.join(so['data'], joincols, 'inner').select(joincols)
    
if debug:
    display(common)

In [0]:
# Row counts by source for missing from superset
x = superset.join(common, joincols, "left_anti")
print(f'{"common":25} {x.count()} tables missing from superset')

for so in jsons:
    x = superset.join(so['data'], joincols, "left_anti")
    print(f'{so["name"]:25} {x.count()} tables missing from superset')

    if debug:
        display(x.take(7))

## Superset with flags for which source includes the table

In [0]:
census = superset.join(jsons[0]['data'].withColumnRenamed('num_rows', jsons[0]['name']), joincols, 'left_outer')

for so in jsons[1:]:
    census = census.join(so['data'].withColumnRenamed('num_rows', so['name']), joincols, 'left_outer')

if debug:
    display(census.take(20))

In [0]:
# Three rows - counts where: 
#   num_rows is null    object doesn't exist in that environment
#   num_rows == -1      object exists, but no quick row count available
#   num_rows >= 0       object exists and estimated row count is available

display(census.groupby('object_type').agg(fn.sum(fn.when(fn.isnull(census.dev1_e2e), 1).otherwise(0)).alias('dev1_e2e'), \
    fn.sum(fn.when(fn.isnull(census.dev3_dress_rehearsal), 1).otherwise(0)).alias('dev3_dress_rehearsal'), \
    fn.sum(fn.when(fn.isnull(census.prd1_prod), 1).otherwise(0)).alias('prd1_prod'), 
    fn.sum(fn.when(fn.isnull(census.stg1_dev), 1).otherwise(0)).alias('stg1_dev'), 
    ))

### Get list of tables we have in bronze & steel (mfcs tables we consume)

In [0]:
# Bronze tables
ucb = spark.sql(f'''select table_catalog, table_schema, table_name
               from system.information_schema.tables 
               where table_catalog = "bronze_{env}"
                  and table_schema = "mfcs"
                  and table_name not like "__material%"
                  and table_name not like "event_log_%"
                  ''')

# Steel tables
ucs = spark.sql(f'''select table_catalog, table_schema, table_name
               from system.information_schema.tables 
               where table_catalog = "steel_{env}"
                  and table_schema = "mfcs"
                  and table_name not like "__material%"
                  and table_name not like "event_log_%"
                  ''')

# Check if there are tables in bronze or steel, but not both
if ucb.join(ucs, ['table_name'], "anti").count() > 0:
    print('**** WARNING: Different tables exist between bronze and steel for MFCS! *****')

# Get the 
tablesWeHave = ucb.join(ucs, ['table_name'], "inner").distinct().select(ucb.table_name.alias('obj'))

# Show your work if asked
if debug:
    display(tablesWeHave)

## Compare common list with tables we have & list those we don't consume

In [0]:
debug = True
# Objects in common among all known environments, but we're not consuming them
missing = common.join(tablesWeHave, [fn.lower(tablesWeHave.obj) == fn.lower(common.obj)], "left_anti")

# Objects we consume (and expect to have!) but do not exist everywhere
# First join with census, then drop the duplicate obj column from tablesWeHave before the second join
uhoh = tablesWeHave.join(census, [fn.lower(tablesWeHave.obj) == fn.lower(census.obj)], "left").drop(tablesWeHave.obj).join(common, [fn.lower(census.obj) == fn.lower(common.obj)], "left_anti")

# Uh-oh!  If there are any tables we consume that are NOT in the common list, we gotta know now:
if uhoh.count() > 0:
    display(uhoh.withColumn('Not universally available!', fn.lit('Warning')))

if debug:
    display(census)
    display(uhoh)

In [0]:
dep = spark.sql(""" select  target_catalog, target_schema, target_table, source_catalog, source_schema, source_table 
                    from    metadata_dev.control.table_hierarchy
                    """)

display(dep.join(uhoh, [(fn.lower(uhoh.obj) == dep.target_table) | (fn.lower(uhoh.obj) == dep.source_table)]).select(uhoh.columns + ['source_catalog', 'source_schema', 'source_table', 'target_catalog', 'target_schema', 'target_table', ]))